# Plataforma de codificación Polar con CRC y bloque largo para redes satelitales

Este cuaderno constituye una plataforma educativa en **Codificación Polar** y **Modelamiento de canal satelital** implementada sobre **TensorFlow** y **NVIDIA Sionna** para presentación de una tesis de posgrado en telecomunicaciones de la Universidad Nacional de Ingeniera - Perú.

A diferencia de los códigos algebraicos clásicos (e.g., Reed-Solomon) o de las construcciones universales aproximadas, el conjunto óptimo de bits congelados $\mathcal{F}$ de un **código Polar** depende explícitamente de las estadísticas del canal físico, de la modulación y del receptor empleado.

### Convenciones Matemáticas y Notación del Cuaderno

| Magnitud / Símbolo | Convención adoptada | Nota clave de implementación |
| :--- | :--- | :--- |
| **Ruido complejo** | $n \sim \mathcal{CN}(0, N_0)$ | Componentes I/Q independientes $\mathcal{N}(0, N_0/2)$ |
| **Potencia de canal** | $\mathbb{E}[\lvert h \rvert^2] = 1$ | Garantiza comparación justa en $E_b/N_0$ entre canales |
| **Relación $E_b/N_0 \leftrightarrow N_0$** | $N_0 = \left(R_{\text{efec}} \cdot m \cdot 10^{(E_b/N_0)_{\text{dB}}/10}\right)^{-1}$ | Con energía media de símbolo $E_s = 1$, $m = \log_2 M$ bits/símb |
| **LLR en Sionna** | $L_{\text{Sionna}}(b) = \ln \frac{P(b=1 \mid y)}{P(b=0 \mid y)}$ | $L > 0 \Rightarrow \hat{b} = 1$; acierto si $(2b-1)L > 0$ |
| **LLR en Arıkan (SC)** | $L_{\text{Arıkan}}(u) = \ln \frac{P(u=0 \mid y)}{P(u=1 \mid y)} = -L_{\text{Sionna}}$ | El motor Genie-Aided SC invierte el signo al entrar a la mariposa |
| **Bloque de coherencia** | $L_c = \text{bloq\_coher} \in [1, N]$ | $h$ constante durante $L_c$ símbolos consecutivos ($L_c \ge N/m$, e.g. $L_c=N$: plano) |

> **Nota sobre hardware y reproducibilidad:** En la carpeta `resources/` se incluyen los archivos `.yml` para reproducir el entorno Conda (`sionna` con aceleración GPU NVIDIA o CPU). En Google Colab, ejecute la celda del Módulo 0 dos veces si es necesaria la instalación automática de Sionna.

---
## Módulo 0: Configuración del Entorno y Reproducibilidad

Configura el dispositivo de cómputo (`CUDA_VISIBLE_DEVICES`), habilita el crecimiento dinámico de VRAM (`set_memory_growth`) para evitar reservas monolíticas en GPU y carga las librerías base (`sionna`, `tensorflow`, `numpy`, `pandas`, `matplotlib`, `psutil`). Si desea forzar ejecución en CPU en una máquina con GPU, cambiar `gpu_num = ""`.

In [ ]:
# Configuración del dispositivo (GPU si está disponible; cambiar gpu_num = "" para forzar CPU)
import os

if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Importa Sionna (con soporte de auto-instalación en Google Colab)
try:
    import sionna
except ImportError as e:
    import sys
    if "google.colab" in sys.modules:
        print("Instalando Sionna en Google Colab...")
        print("⚠️ OJO: Al finalizar el reinicio del entorno, EJECUTE ESTA CELDA DE NUEVO.")
        os.system("pip install sionna")
        os.kill(os.getpid(), 5)
    else:
        raise e

# Configuración segura de TensorFlow para GPUs detectadas
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            print(f"Advertencia al configurar memoria GPU: {exc}")
    try:
        details = tf.config.experimental.get_device_details(gpus[0])
        device_name = details.get("device_name", gpus[0].name)
    except Exception:
        device_name = gpus[0].name
    HAS_GPU = True
    print(f"✅ Dispositivo activo: GPU ({device_name})")
else:
    HAS_GPU = False
    print("ℹ️ Dispositivo activo: CPU (no se detectó GPU NVIDIA)")
tf.get_logger().setLevel("ERROR")

# Librerías científicas, persistencia y monitoreo de recursos
import gc
import json
import time
from datetime import datetime

import numpy as np
import pandas as pd
import psutil

%matplotlib inline
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.55,
    "font.size": 10,
})

---
## Módulo 1: Parametrización del Experimento y Restricciones de Diseño

Centraliza todos los parámetros de la plataforma. Antes de instanciar cualquier bloque, la celda valida las restricciones algebraicas y de Sionna:

- **Longitud de palabra código ($N = 2^n$):** La transformada polar de Arıkan $G_N = F^{\otimes n}$ exige potencia de 2.
- **Alineación con la modulación ($m = \text{bitsxsimb}$):** Tanto $N$ como $K$ (para la línea base `SinCodigo`) deben ser múltiplos de $m$. En particular, 64-QAM ($m=6$) no divide ninguna potencia de 2 sin *rate-matching*.
- **Tasa efectiva con CRC:** Al concatenar un CRC de longitud $L_{\text{CRC}}$ bits (e.g., `CRC24A` con $L_{\text{CRC}}=24$), el codificador Polar protege $K$ bits totales pero solo $K_{\text{efec}} = K - L_{\text{CRC}}$ bits son información útil, por lo que $R_{\text{efec}} = (K - L_{\text{CRC}})/N$ y $E_b/N_0$ debe referirse a los bits útiles.
- **Resolución estadística Monte Carlo:** Con $M = \text{lote\_mc} \times \text{num\_iter}$ tramas por bit-canal, el piso de resolución empírico es $\text{BER}_{\min} = 1/M$ y la desviación estándar binomial de cada estimador es $\sigma_{\widehat{\text{BER}}_i} = \sqrt{p_i(1-p_i)/M}$.
- **Error de estimación CSI ($\sigma_e^2 = \text{csi\_err\_var}$):** Modela $\hat{h} = h + e_h$ con $e_h \sim \mathcal{CN}(0, \sigma_e^2)$ por bloque de coherencia (`0.0` = CSI perfecto; `0.01` $= -20\text{ dB}$; `0.05` $\approx -13\text{ dB}$; `0.10` $= -10\text{ dB}$).

In [ ]:
# =====================================================================
# PARÁMETROS GENERALES DE LA SIMULACIÓN
# =====================================================================

# 1. Parámetros básicos del código y modulación
SEED = 0                 # Semilla aleatoria para reproducibilidad (Sionna e interleaver)
sionna.phy.config.seed = SEED
N = 512                  # Longitud del código Polar (debe ser potencia de 2)
K = 256                  # Bits de información (1 <= K < N; tasa nominal R = K/N)
bitsxsimb = 1            # Bits por símbolo: 1 (BPSK/PAM), 2 (QPSK/QAM), 4 (16-QAM)
modulacion = "pam" if bitsxsimb == 1 else "qam"

# 2. Parámetros del canal inalámbrico
# Canales válidos: "awgn", "rayleigh", "rice", "lognormal", "suzuki",
#                  "nakagami", "shadowed_rician", "impulsive"
canal = "awgn"
bloq_coher = N // 16     # Símbolos con ganancia h constante (use N para desvanecimiento plano)
k_factor_db = 6.0        # Rice / Shadowed-Rician: factor K en dB (típico: 4 a 15 dB)
sigma_db = 6.0           # Lognormal / Suzuki: desv. estándar del sombreado [dB] (2 leve, 6 severo)
m_nakagami = 1.5         # Nakagami-m / Shadowed-Rician: parámetro de forma (m >= 0.5)
imp_prob = 0.05          # Impulsive: probabilidad p de impulso por símbolo (0.01 a 0.10)
imp_gamma = 50.0         # Impulsive: razón de potencia impulso / fondo (potencia total = N0)

# 3. Parámetros del receptor
ecualizador = "no_eq"    # "no_eq" (solo AWGN/Impulsive), "zf" o "mmse"
imp_demapper = "mixture" # Impulsive: "mixture" (APP exacto) o "gaussian" (desajustado)
csi_err_var = 0.0        # Varianza error CSI sigma_e^2 (0.0 = CSI perfecto)
list_scl = 8             # Tamaño de lista L para decodificador Polar SCL / CA-SCL
LLR_clip = 500.0         # Recorte simétrico [-LLR_clip, +LLR_clip] para estabilidad numérica

# 4. Parámetros de construcción Monte Carlo (cada bit-canal recibe lote_mc * num_iter muestras)
mc_modo = "genie_sc"     # "genie_sc" (vectorizado O(N log N), recomendado) o "por_bit_canal" (legado O(N^2 log N))
lote_mc = 1000           # Tramas en paralelo por paso (controla uso de RAM/VRAM)
num_iter = 1000          # Lotes secuenciales acumulados (1000 x 1000 = 1,000,000 muestras/bit-canal)
snr_simulacion = -1.2    # Eb/N0 [dB] de diseño inicial (puede recalcularse en el Módulo 4)
# Valores recomendados de Eb/N0 [dB] por canal (parámetros: R=0.5, BPSK, CSI perfecto):
#   - awgn            : -1.2 dB  |  - rayleigh        :  1.0 dB
#   - rice            : -0.3 dB  |  - lognormal       :  2.4 dB
#   - suzuki          :  5.0 dB  |  - nakagami        :  0.3 dB
#   - shadowed_rician :  0.5 dB  |  - impulsive       : -5.7 dB (mixture) / -1.9 dB (gaussian)

# 5. Parámetros de CRC (3GPP TS 38.212 para CA-SCL)
crc_degree = "CRC24A"    # Opciones 5G NR: "CRC24A", "CRC24B", "CRC24C", "CRC16", "CRC11", "CRC6"
crc_codif = sionna.phy.fec.crc.CRCEncoder(crc_degree=crc_degree)
crc_long = crc_codif.crc_length

# 6. Parámetros del barrido BER/BLER vs Eb/N0
EBNO_DB_MIN = 0.0        # Eb/N0 mínimo [dB]
EBNO_DB_MAX = 12.0       # Eb/N0 máximo [dB] (use ~20 dB en canales con desvanecimiento severo)
EBNO_RANGO = 13          # Cantidad de puntos equidistantes en [EBNO_DB_MIN, EBNO_DB_MAX]
lote_berplot = 200       # Palabras código por lote en las curvas BER
TARGET_BLOCK_ERRORS = 250  # Criterio de parada temprana por punto de Eb/N0
MAX_MC_ITER_BER = 50     # Límite máximo de lotes por punto de Eb/N0

# =====================================================================
# VALIDACIÓN TEMPRANA DE RESTRICCIONES
# =====================================================================
CANALES_VALIDOS = {
    "awgn", "rayleigh", "rice", "lognormal",
    "suzuki", "nakagami", "shadowed_rician", "impulsive",
}
CANALES_SIN_FADING = {"awgn", "impulsive"}
ECUALIZADORES_VALIDOS = {"no_eq", "zf", "mmse"}
IMP_DEMAPPERS_VALIDOS = {"mixture", "gaussian"}
MC_MODOS_VALIDOS = {"genie_sc", "por_bit_canal"}

assert canal in CANALES_VALIDOS, f"Canal inválido '{canal}'. Opciones: {sorted(CANALES_VALIDOS)}"
assert ecualizador in ECUALIZADORES_VALIDOS, f"Ecualizador inválido '{ecualizador}'."
assert imp_demapper in IMP_DEMAPPERS_VALIDOS, f"Demapeo impulsivo inválido '{imp_demapper}'."
assert mc_modo in MC_MODOS_VALIDOS, f"mc_modo inválido '{mc_modo}'. Opciones: {sorted(MC_MODOS_VALIDOS)}"
assert csi_err_var >= 0.0, f"csi_err_var ({csi_err_var}) debe ser >= 0.0."
assert m_nakagami >= 0.5, f"m_nakagami ({m_nakagami}) debe cumplir m >= 0.5."
assert 0.0 < imp_prob < 1.0, f"imp_prob ({imp_prob}) debe estar en (0, 1)."
assert imp_gamma >= 0.0, f"imp_gamma ({imp_gamma}) debe ser >= 0.0."
assert not (canal not in CANALES_SIN_FADING and ecualizador == "no_eq"), (
    "⚠️ 'no_eq' solo es válido para canales sin desvanecimiento ('awgn', 'impulsive'); use 'zf' o 'mmse'."
)
assert N > 0 and (N & (N - 1)) == 0, f"N ({N}) debe ser potencia de 2 para códigos Polar."
assert 1 <= K < N, f"K ({K}) debe cumplir 1 <= K < N ({N})."
assert N % bitsxsimb == 0, f"N ({N}) debe ser divisible por bitsxsimb ({bitsxsimb})."
assert K % bitsxsimb == 0, f"K ({K}) debe ser divisible por bitsxsimb ({bitsxsimb}) para SinCodigo."
assert canal in CANALES_SIN_FADING or (1 <= bloq_coher <= N), (
    f"bloq_coher ({bloq_coher}) debe estar entre 1 y N ({N})."
)
assert EBNO_DB_MIN < EBNO_DB_MAX and EBNO_RANGO >= 2, "Rango de Eb/N0 inválido."

if canal in CANALES_SIN_FADING and csi_err_var > 0.0:
    print(f"ℹ️ Aviso: csi_err_var={csi_err_var} se ignora en el canal '{canal}' (h = 1 sin desvanecimiento).")
if K <= crc_long:
    print(f"⚠️ Aviso: K ({K}) <= crc_long ({crc_long}); la curva Polar + {crc_degree} se omitirá.")


# =====================================================================
# BLOQUES COMUNES DE SIONNA Y UTILIDAD DE CONFIGURACIÓN
# =====================================================================
def build_constellation_bit_masks(num_bits_per_symbol):
    """Construye máscaras booleanas [m, 2^m] para bits 1 y 0 (orden MSB-first de Sionna)."""
    num_points = 1 << num_bits_per_symbol
    indices = np.arange(num_points, dtype=np.int32)
    shifts = np.arange(num_bits_per_symbol - 1, -1, -1, dtype=np.int32)[:, None]
    bits_table = ((indices[None, :] >> shifts) & 1).astype(bool)
    return tf.constant(bits_table, dtype=tf.bool), tf.constant(~bits_table, dtype=tf.bool)


datos = sionna.phy.mapping.BinarySource()
constelacion = sionna.phy.mapping.Constellation(modulacion, bitsxsimb)
constellation_points = tf.cast(constelacion.points, tf.complex64)
bit_masks_1, bit_masks_0 = build_constellation_bit_masks(bitsxsimb)
mapeador = sionna.phy.mapping.Mapper(constellation=constelacion)
demapeador = sionna.phy.mapping.Demapper("app", constellation=constelacion)
interleaver = sionna.phy.fec.interleaving.RandomInterleaver(seed=SEED, axis=-1)
deinterleaver = sionna.phy.fec.interleaving.Deinterleaver(interleaver)


def get_sim_params():
    """Devuelve una instantánea consistente de los parámetros activos del cuaderno."""
    return {
        "seed": int(SEED),
        "N": int(N),
        "K": int(K),
        "bitsxsimb": int(bitsxsimb),
        "crc_degree": str(crc_degree),
        "canal": str(canal),
        "bloq_coher": int(bloq_coher),
        "k_factor_db": float(k_factor_db),
        "sigma_db": float(sigma_db),
        "m_nakagami": float(m_nakagami),
        "imp_prob": float(imp_prob),
        "imp_gamma": float(imp_gamma),
        "imp_demapper": str(imp_demapper),
        "ecualizador": str(ecualizador),
        "csi_err_var": float(csi_err_var),
        "list_scl": int(list_scl),
        "LLR_clip": float(LLR_clip),
        "lote_mc": int(lote_mc),
        "num_iter": int(num_iter),
        "snr_simulacion": float(snr_simulacion),
        "mc_modo": str(mc_modo),
    }


total_mc_samples = lote_mc * num_iter
r_eff_crc = (K - crc_long) / N if K > crc_long else 0.0
print(
    f"✅ Parámetros validados | N={N}, K={K} (R={K/N:.3f}, R_efec_CRC={r_eff_crc:.3f}) | "
    f"Mod={modulacion.upper()} ({bitsxsimb} b/símb)\n"
    f"   Canal={canal} (Lc={bloq_coher}) | Eq={ecualizador} (sigma_e^2={csi_err_var}) | "
    f"MC[{mc_modo}]={total_mc_samples:,} muestras/bit-canal (BER_min={1.0/total_mc_samples:.1e})"
)

---
## Módulo 2: Modelos de Canal Inalámbrico y Verificación Estadística

Se implementan **ocho modelos de canal** compilados con XLA (`@tf.function(jit_compile=True)`). Todos los canales con desvanecimiento satisfacen la normalización de potencia unitaria $\mathbb{E}[|h|^2] = 1$ y operan en modo **block-fading** (ganancia $h$ constante durante $L_c = \text{bloq\_coher}$ símbolos consecutivos):

1. **AWGN:** $y = x + n$, con $n \sim \mathcal{CN}(0, N_0)$ y $h = 1$.
2. **Rayleigh (NLOS):** $h \sim \mathcal{CN}(0, 1)$, de modo que $|h| \sim \text{Rayleigh}(1/\sqrt{2})$ y $|h|^2 \sim \text{Exp}(1)$.
3. **Rice (LOS + NLOS):**
   $$h = \sqrt{\frac{K_{\text{lin}}}{K_{\text{lin}}+1}}\,e^{j\phi} + \sqrt{\frac{1}{K_{\text{lin}}+1}}\,g, \quad \phi \sim \mathcal{U}[0, 2\pi), \; g \sim \mathcal{CN}(0, 1), \; K_{\text{lin}} = 10^{K_{\text{dB}}/10}$$
4. **Log-Normal (sombreado lento):** $h = e^{z}$ con $z \sim \mathcal{N}(\mu_{\text{LN}}, \sigma_{\text{lin}}^2)$ y $\sigma_{\text{lin}} = \frac{\ln 10}{20}\,\sigma_{\text{dB}}$. Como $\mathbb{E}[e^{2z}] = \exp(2\mu_{\text{LN}} + 2\sigma_{\text{lin}}^2)$, imponer potencia unitaria exige exactamente:
   $$\mu_{\text{LN}} = -\sigma_{\text{lin}}^2$$
5. **Suzuki (Rayleigh $\times$ Log-Normal):** $h = g \cdot e^z$ con $g \sim \mathcal{CN}(0,1)$ independiente del sombreado $e^z$ (desvanecimiento rápido y lento combinados).
6. **Nakagami-$m$ ($m \ge 0.5$):** $|h|^2 \sim \text{Gamma}(\text{forma } m,\, \text{tasa } m)$ (media unitaria) y $\phi \sim \mathcal{U}[0, 2\pi)$, generando muestras Gamma en XLA mediante rechazo de **Marsaglia–Tsang** con reducción exacta $\text{Gamma}(m) = \text{Gamma}(m+1)\cdot U^{1/m}$ para $0.5 \le m < 1$. Note que $m=1$ recupera Rayleigh y $m \to \infty$ tiende a AWGN.
7. **Shadowed-Rician (modelo de Abdi — Satelital LMS / 5G NTN):** Combina una componente LOS sombreada por Nakagami-$m$ ($p_{\text{sh}} \sim \text{Gamma}(\text{forma } m,\, \text{tasa } m)$) y multitrayecto NLOS Rayleigh:
   $$h = \sqrt{\frac{K_{\text{lin}}}{K_{\text{lin}}+1}\,p_{\text{sh}}}\,e^{j\phi} + \sqrt{\frac{1}{K_{\text{lin}}+1}}\,g$$
8. **AWGN con Ruido Impulsivo (Bernoulli-Gaussian):** $y = x + n_{\text{bg}} + b \cdot n_{\text{imp}}$, donde $b \sim \text{Bernoulli}(p)$, $n_{\text{bg}} \sim \mathcal{CN}(0, \sigma_0^2)$ y $n_{\text{imp}} \sim \mathcal{CN}(0, \Gamma \sigma_0^2)$. Para mantener la potencia total de ruido en $N_0$:
   $$\sigma_0^2 = \frac{N_0}{1 + p\Gamma}, \qquad \sigma_1^2 = (1 + \Gamma)\sigma_0^2$$

> **Pregunta de reflexión 2.1:** En el canal Log-Normal, ¿qué error sistemático se introduciría en las curvas BER vs. $E_b/N_0$ si se fijara $\mu_{\text{LN}} = 0$ en lugar de $\mu_{\text{LN}} = -\sigma_{\text{lin}}^2$ para $\sigma_{\text{dB}} = 6\text{ dB}$? Calcule el desplazamiento resultante en dB.

In [ ]:
# =====================================================================
# MODELOS DE CANAL COMPILADOS CON XLA
# =====================================================================

def _sample_complex_normal(shape, variance):
    """Genera muestras complejas circularmente simétricas CN(0, variance)."""
    stddev = tf.sqrt(tf.cast(variance, tf.float32) / 2.0)
    re = sionna.phy.config.tf_rng.normal(shape, mean=0.0, stddev=stddev, dtype=tf.float32)
    im = sionna.phy.config.tf_rng.normal(shape, mean=0.0, stddev=stddev, dtype=tf.float32)
    return tf.complex(re, im)


def _sample_unit_phasor(shape):
    """Genera fasores de amplitud unitaria y fase uniforme en [0, 2*pi)."""
    phi = sionna.phy.config.tf_rng.uniform(
        shape, minval=0.0, maxval=tf.constant(2.0 * np.pi, tf.float32), dtype=tf.float32
    )
    return tf.complex(tf.cos(phi), tf.sin(phi))


def _sample_lognormal_amplitude(shape, sigma_db):
    """Genera amplitudes lognormales e^z normalizadas a potencia unitaria E[e^(2z)] = 1."""
    sigma_lin = tf.cast(tf.math.log(10.0) / 20.0 * sigma_db, tf.float32)
    mu_ln = -tf.square(sigma_lin)
    z = sionna.phy.config.tf_rng.normal(shape, mean=mu_ln, stddev=sigma_lin, dtype=tf.float32)
    return tf.exp(z)


def _fading_repetitions(n_sym, bloq_coher):
    """Cantidad mínima de bloques de coherencia para cubrir n_sym símbolos."""
    return tf.cast(tf.math.floordiv(n_sym + bloq_coher - 1, bloq_coher), tf.int32)


def _expand_coherence_blocks(h_block, bloq_coher, n_sym):
    """Repite cada bloque de coherencia sobre bloq_coher símbolos y recorta a n_sym."""
    return tf.repeat(h_block, repeats=bloq_coher, axis=1)[:, :n_sym]


def _prepare_fading_context(x, bloq_coher, ebno_db, bitsxsimb, coderate):
    """Prepara dimensiones, bloques de coherencia, varianza N0 y ruido AWGN."""
    batch_size = tf.shape(x)[0]
    n_sym = tf.shape(x)[-1]
    bloq_c = tf.cast(bloq_coher, tf.int32)
    no = sionna.phy.utils.ebnodb2no(ebno_db, bitsxsimb, coderate=coderate)
    rep = _fading_repetitions(n_sym, bloq_c)
    awgn = _sample_complex_normal(tf.shape(x), no)
    return batch_size, n_sym, bloq_c, rep, no, awgn


def _sample_gamma_unit_mean(shape, m_param):
    """Genera muestras Gamma(m, m) de media unitaria compatibles con XLA (Marsaglia-Tsang)."""
    m_tf = tf.cast(m_param, tf.float32)
    alpha = tf.where(m_tf < 1.0, m_tf + 1.0, m_tf)
    d = alpha - (1.0 / 3.0)
    c = tf.math.rsqrt(9.0 * d)

    accepted = tf.zeros(shape, dtype=tf.bool)
    samples = tf.ones(shape, dtype=tf.float32)

    def cond(acc, _samp):
        return tf.reduce_any(tf.logical_not(acc))

    def body(acc, samp):
        z = sionna.phy.config.tf_rng.normal(shape, mean=0.0, stddev=1.0, dtype=tf.float32)
        u = sionna.phy.config.tf_rng.uniform(shape, minval=1e-7, maxval=1.0, dtype=tf.float32)
        v_base = 1.0 + c * z
        v = v_base * v_base * v_base
        v_safe = tf.maximum(v, 1e-7)
        accept_cond = tf.logical_and(
            v_base > 0.0,
            tf.math.log(u) < (0.5 * z * z + d * (1.0 - v_safe + tf.math.log(v_safe))),
        )
        newly_acc = tf.logical_and(tf.logical_not(acc), accept_cond)
        return tf.logical_or(acc, newly_acc), tf.where(newly_acc, d * v_safe, samp)

    _, samples = tf.while_loop(cond, body, [accepted, samples])
    # Reducción exacta Gamma(m) = Gamma(m + 1) * U^(1/m) cuando 0.5 <= m < 1
    u_boost = sionna.phy.config.tf_rng.uniform(shape, minval=1e-7, maxval=1.0, dtype=tf.float32)
    samples = tf.where(m_tf < 1.0, samples * tf.pow(u_boost, 1.0 / m_tf), samples)
    return samples / m_tf


@tf.function(jit_compile=True)
def canalAWGN(ebno_db, bitsxsimb, coderate, x):
    no = sionna.phy.utils.ebnodb2no(ebno_db, bitsxsimb, coderate=coderate)
    awgn = _sample_complex_normal(tf.shape(x), no)
    h = tf.complex(1.0, 0.0)
    return x + awgn, h, no


@tf.function(jit_compile=True)
def canalRayleigh(bloq_coher, ebno_db, bitsxsimb, coderate, x):
    batch_size, n_sym, bloq_c, rep, no, awgn = _prepare_fading_context(x, bloq_coher, ebno_db, bitsxsimb, coderate)
    h_blk = _sample_complex_normal([batch_size, rep], 1.0)
    h = _expand_coherence_blocks(h_blk, bloq_c, n_sym)
    return h * x + awgn, h, no


@tf.function(jit_compile=True)
def canalRice(bloq_coher, k_factor_db, ebno_db, bitsxsimb, coderate, x):
    batch_size, n_sym, bloq_c, rep, no, awgn = _prepare_fading_context(x, bloq_coher, ebno_db, bitsxsimb, coderate)
    k_lin = tf.pow(10.0, tf.cast(k_factor_db, tf.float32) / 10.0)
    a_los = tf.cast(tf.sqrt(k_lin / (k_lin + 1.0)), tf.complex64)
    b_nlos = tf.cast(tf.sqrt(1.0 / (k_lin + 1.0)), tf.complex64)
    h_blk = a_los * _sample_unit_phasor([batch_size, rep]) + b_nlos * _sample_complex_normal([batch_size, rep], 1.0)
    h = _expand_coherence_blocks(h_blk, bloq_c, n_sym)
    return h * x + awgn, h, no


@tf.function(jit_compile=True)
def canalLognormal(bloq_coher, sigma_db, ebno_db, bitsxsimb, coderate, x):
    batch_size, n_sym, bloq_c, rep, no, awgn = _prepare_fading_context(x, bloq_coher, ebno_db, bitsxsimb, coderate)
    amp = _sample_lognormal_amplitude([batch_size, rep], sigma_db)
    h = _expand_coherence_blocks(tf.complex(amp, tf.zeros_like(amp)), bloq_c, n_sym)
    return h * x + awgn, h, no


@tf.function(jit_compile=True)
def canalRayleighLognormal(bloq_coher, sigma_db, ebno_db, bitsxsimb, coderate, x):
    batch_size, n_sym, bloq_c, rep, no, awgn = _prepare_fading_context(x, bloq_coher, ebno_db, bitsxsimb, coderate)
    g = _sample_complex_normal([batch_size, rep], 1.0)
    amp = _sample_lognormal_amplitude([batch_size, rep], sigma_db)
    h = _expand_coherence_blocks(g * tf.complex(amp, tf.zeros_like(amp)), bloq_c, n_sym)
    return h * x + awgn, h, no


@tf.function(jit_compile=True)
def canalNakagami(bloq_coher, m_nakagami, ebno_db, bitsxsimb, coderate, x):
    batch_size, n_sym, bloq_c, rep, no, awgn = _prepare_fading_context(x, bloq_coher, ebno_db, bitsxsimb, coderate)
    potencia = _sample_gamma_unit_mean([batch_size, rep], m_nakagami)
    amp = tf.cast(tf.sqrt(tf.maximum(potencia, 1e-12)), tf.complex64)
    h = _expand_coherence_blocks(amp * _sample_unit_phasor([batch_size, rep]), bloq_c, n_sym)
    return h * x + awgn, h, no


@tf.function(jit_compile=True)
def canalShadowedRician(bloq_coher, k_factor_db, m_nakagami, ebno_db, bitsxsimb, coderate, x):
    batch_size, n_sym, bloq_c, rep, no, awgn = _prepare_fading_context(x, bloq_coher, ebno_db, bitsxsimb, coderate)
    k_lin = tf.pow(10.0, tf.cast(k_factor_db, tf.float32) / 10.0)
    p_los = k_lin / (k_lin + 1.0)
    p_nlos = 1.0 / (k_lin + 1.0)

    sombreado_los = _sample_gamma_unit_mean([batch_size, rep], m_nakagami)
    amp_los = tf.cast(tf.sqrt(tf.maximum(p_los * sombreado_los, 1e-12)), tf.complex64)
    h_los = amp_los * _sample_unit_phasor([batch_size, rep])
    h_nlos = _sample_complex_normal([batch_size, rep], p_nlos)
    h = _expand_coherence_blocks(h_los + h_nlos, bloq_c, n_sym)
    return h * x + awgn, h, no


@tf.function(jit_compile=True)
def canalImpulsivo(imp_prob, imp_gamma, ebno_db, bitsxsimb, coderate, x):
    no = sionna.phy.utils.ebnodb2no(ebno_db, bitsxsimb, coderate=coderate)
    p_tf = tf.cast(imp_prob, tf.float32)
    gamma_tf = tf.cast(imp_gamma, tf.float32)

    no_bg = no / (1.0 + p_tf * gamma_tf)
    ruido_bg = _sample_complex_normal(tf.shape(x), no_bg)
    ruido_imp = _sample_complex_normal(tf.shape(x), gamma_tf * no_bg)

    u = sionna.phy.config.tf_rng.uniform(tf.shape(x), minval=0.0, maxval=1.0, dtype=tf.float32)
    mascara = tf.cast(u < p_tf, tf.complex64)
    h = tf.complex(1.0, 0.0)
    return x + ruido_bg + mascara * ruido_imp, h, no


def sample_active_channel(x, ebno_db, coderate, params):
    """Despachador único de canal a partir del diccionario de parámetros."""
    c = params["canal"]
    if c == "awgn":
        return canalAWGN(ebno_db, params["bitsxsimb"], coderate, x)
    if c == "rayleigh":
        return canalRayleigh(params["bloq_coher"], ebno_db, params["bitsxsimb"], coderate, x)
    if c == "rice":
        return canalRice(params["bloq_coher"], params["k_factor_db"], ebno_db, params["bitsxsimb"], coderate, x)
    if c == "lognormal":
        return canalLognormal(params["bloq_coher"], params["sigma_db"], ebno_db, params["bitsxsimb"], coderate, x)
    if c == "suzuki":
        return canalRayleighLognormal(params["bloq_coher"], params["sigma_db"], ebno_db, params["bitsxsimb"], coderate, x)
    if c == "nakagami":
        return canalNakagami(params["bloq_coher"], params["m_nakagami"], ebno_db, params["bitsxsimb"], coderate, x)
    if c == "shadowed_rician":
        return canalShadowedRician(
            params["bloq_coher"], params["k_factor_db"], params["m_nakagami"],
            ebno_db, params["bitsxsimb"], coderate, x
        )
    if c == "impulsive":
        return canalImpulsivo(params["imp_prob"], params["imp_gamma"], ebno_db, params["bitsxsimb"], coderate, x)
    raise ValueError(f"Canal no reconocido: {c}")

In [ ]:
# Diagnóstico empírico del canal activo: verifica E[|h|^2] ≈ 1 y visualiza la estadística física
EJECUTAR_DIAGNOSTICO_CANAL = True

if EJECUTAR_DIAGNOSTICO_CANAL:
    p_diag = get_sim_params()
    ebno_probe = tf.constant(5.0, dtype=tf.float32)
    n_sym_probe = max(1, p_diag["N"] // p_diag["bitsxsimb"])
    bloq_per_frame = max(1, (n_sym_probe + p_diag["bloq_coher"] - 1) // p_diag["bloq_coher"])
    n_diag_batch = 512 if p_diag["canal"] in CANALES_SIN_FADING else min(4096, max(512, 8192 // bloq_per_frame))
    x_probe = tf.complex(tf.ones([n_diag_batch, n_sym_probe], dtype=tf.float32), tf.zeros([n_diag_batch, n_sym_probe], dtype=tf.float32))
    y_probe, h_probe, no_probe = sample_active_channel(x_probe, ebno_probe, coderate=p_diag["K"] / p_diag["N"], params=p_diag)

    n_emp = (y_probe - h_probe * x_probe).numpy().ravel()
    no_teo = float(no_probe.numpy())
    no_emp = float(np.mean(np.abs(n_emp) ** 2))

    if p_diag["canal"] not in CANALES_SIN_FADING:
        h_vals = h_probe.numpy()[:, ::p_diag["bloq_coher"]].ravel()
        amp = np.abs(h_vals)
        pot_emp = float(np.mean(amp ** 2))
        print(
            f"📊 Diagnóstico Canal '{p_diag['canal']}' | E[|h|^2] empírica = {pot_emp:.4f} (teórica = 1.0000) | "
            f"E[|n|^2] = {no_emp:.4f} vs N0 = {no_teo:.4f}"
        )
        fig, ax = plt.subplots(figsize=(8, 4))
        upper = float(np.percentile(amp, 99.5))
        ax.hist(amp[amp <= upper], bins=70, density=True, alpha=0.7, color="#2b6cb0", label=f"Empírica ({p_diag['canal']})")
        r_grid = np.linspace(0.0, upper, 300)
        ax.plot(r_grid, 2.0 * r_grid * np.exp(-r_grid ** 2), "r--", lw=1.8, label="Referencia Rayleigh (potencia unitaria)")
        ax.set_title(f"Distribución empírica de la envolvente |h| — Canal {p_diag['canal']}")
        ax.set_xlabel("|h|")
        ax.set_ylabel("Densidad de probabilidad f(|h|)")
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        n_re = np.real(n_emp)
        print(
            f"📊 Diagnóstico Canal '{p_diag['canal']}' | h = 1.0 | "
            f"E[|n|^2] empírica = {no_emp:.4f} vs N0 teórico = {no_teo:.4f}"
        )
        fig, ax = plt.subplots(figsize=(8, 4))
        lim = float(np.percentile(np.abs(n_re), 99.8))
        ax.hist(n_re[np.abs(n_re) <= lim], bins=90, density=True, alpha=0.7, color="#2b6cb0", label=f"Empírica Re{{n}} ({p_diag['canal']})")
        x_grid = np.linspace(-lim, lim, 400)
        gauss_pdf = np.exp(-x_grid ** 2 / no_teo) / np.sqrt(np.pi * no_teo)
        ax.plot(x_grid, gauss_pdf, "r--", lw=1.8, label="Referencia Gaussiana N(0, N0/2)")
        ax.set_yscale("log")
        ax.set_ylim(bottom=1e-5)
        ax.set_title(f"Colas de la componente real del ruido Re{{n}} (Eb/N0 = 5 dB) — Canal {p_diag['canal']}")
        ax.set_xlabel("Amplitud de ruido Re{n}")
        ax.set_ylabel("Densidad f(Re{n}) [escala log]")
        ax.legend()
        plt.tight_layout()
        plt.show()

---
## Módulo 3: CSI Imperfecto, Ecualización SISO y Demapeo APP

En el receptor, la calidad de la decodificación suave (Polar SCL y LDPC BP) depende críticamente de la **calibración de los LLRs** entregados por la etapa de ecualización y demapeo.

### 3.1 Modelo de Error de Estimación CSI
En canales con desvanecimiento, el receptor observa una estimación ruidosa constante por bloque de coherencia:
$$\hat{h} = h + e_h, \qquad e_h \sim \mathcal{CN}(0, \sigma_e^2)$$
donde $\sigma_e^2 = \text{csi\_err\_var}$ representa el MSE de estimación de pilotos.

### 3.2 Ecualizadores ZF vs. MMSE
1. **Sin ecualización (`no_eq`):** $\hat{x} = y$, $N_{0,\text{eff}} = N_0$ (válido únicamente para `awgn` e `impulsive`).
2. **Zero-Forcing (`zf`):** Invierte ingenuamente $\hat{h}$ ignorando $\sigma_e^2$:
   $$\hat{x}_{\text{ZF}} = \frac{\hat{h}^* y}{|\hat{h}|^2}, \qquad N_{0,\text{eff}}^{\text{ZF}} = \frac{N_0}{|\hat{h}|^2}$$
3. **MMSE (`mmse`):** Incorpora el ruido térmico $N_0$ y la varianza del error CSI $\sigma_e^2$:
   $$g = \frac{\hat{h}^*}{|\hat{h}|^2 + \sigma_e^2 + N_0}, \qquad \hat{x}_{\text{MMSE}} = \frac{g \cdot y}{|g \cdot \hat{h}|}, \qquad N_{0,\text{eff}}^{\text{MMSE}} = \frac{N_0 + \sigma_e^2}{|\hat{h}|^2}$$

> **Observación:** Para un canal SISO escalar de desvanecimiento plano con $\sigma_e^2 = 0$, observe que $g / |g\hat{h}| = \hat{h}^* / |\hat{h}|^2$ y $N_{0,\text{eff}}^{\text{MMSE}} = N_0 / |\hat{h}|^2$. Por tanto, **con CSI perfecto, ZF y MMSE producen exactamente los mismos LLRs blandos** (a diferencia de sistemas MIMO donde existe interferencia entre flujos). Cuando $\sigma_e^2 > 0$, ZF subestima la varianza efectiva del ruido y entrega **LLRs sobreconfiados**, mientras MMSE recalibra $N_{0,\text{eff}}$ evitando degradación severa en SCL y BP.

### 3.3 Demapeo APP Exacto vs. Desajustado en Ruido Impulsivo
Para el canal Bernoulli-Gaussian ($p$, $\Gamma$), la densidad condicional exacta de símbolo es una mezcla de dos gaussianas circulares con varianzas $\sigma_0^2 = N_0/(1+p\Gamma)$ y $\sigma_1^2 = (1+\Gamma)\sigma_0^2$:
$$\ln p(y \mid s) = \operatorname{logsumexp}\!\left(\ln(1-p) - \ln\sigma_0^2 - \frac{|y-s|^2}{\sigma_0^2},\; \ln p - \ln\sigma_1^2 - \frac{|y-s|^2}{\sigma_1^2}\right) - \ln\pi$$
Conservar los términos de normalización $-\ln(\sigma_k^2)$ es indispensable para ponderar correctamente la componente impulsiva. El demapeador exacto (`imp_demapper = "mixture"`) atenúa automáticamente la confianza de los símbolos contaminados por impulsos, mientras que un demapeador gaussiano estándar (`"gaussian"`) asigna magnitudes de LLR desproporcionadas a valores atípicos.

In [ ]:
# =====================================================================
# CSI IMPERFECTO, ECUALIZADORES Y DEMAPEADOR APP DE MEZCLA
# =====================================================================

@tf.function(jit_compile=True)
def aplicar_error_csi(h, bloq_coher, csi_err_var):
    """Añade error de estimación e_h ~ CN(0, sigma_e^2) constante por bloque de coherencia."""
    sigma_e_sq = tf.maximum(tf.cast(csi_err_var, tf.float32), 0.0)
    batch_size = tf.shape(h)[0]
    n_sym = tf.shape(h)[-1]
    bloq_c = tf.cast(bloq_coher, tf.int32)
    rep = _fading_repetitions(n_sym, bloq_c)
    e_h = _sample_complex_normal([batch_size, rep], sigma_e_sq)
    return h + _expand_coherence_blocks(e_h, bloq_c, n_sym)


@tf.function(jit_compile=True)
def sin_ecualizar(y, _h, no):
    return y, no


@tf.function(jit_compile=True)
def zf(y, h, no, eps=1e-12):
    """Zero-Forcing: invierte h_hat directamente (ignora sigma_e^2)."""
    h2 = tf.maximum(tf.math.square(tf.abs(h)), eps)
    x_hat = y * tf.math.conj(h) / tf.cast(h2, tf.complex64)
    no_eff = no / h2
    return x_hat, no_eff


@tf.function(jit_compile=True)
def mmse(y, h, no, csi_err_var=0.0, eps=1e-12):
    """MMSE: incorpora ruido térmico N0 y varianza de error CSI sigma_e^2."""
    sigma_e_sq = tf.maximum(tf.cast(csi_err_var, tf.float32), 0.0)
    h2 = tf.math.square(tf.abs(h))
    g = tf.math.conj(h) / tf.cast(h2 + sigma_e_sq + no, tf.complex64)
    d = g * h
    d_abs = tf.maximum(tf.abs(d), eps)
    x_hat = g * y / tf.cast(d_abs, tf.complex64)
    no_eff = (no + sigma_e_sq) / tf.maximum(h2, eps)
    return x_hat, no_eff


@tf.function(jit_compile=True)
def demap_impulsive_mixture(y, c_points, masks_1, masks_0, no_bg, imp_p, imp_g):
    """Demapeador APP exacto para mezcla Bernoulli-Gaussian (convención Sionna: LLR > 0 => b = 1)."""
    eps = tf.constant(1e-12, dtype=tf.float32)
    p_tf = tf.clip_by_value(tf.cast(imp_p, tf.float32), 0.0, 1.0)
    gamma_tf = tf.maximum(tf.cast(imp_g, tf.float32), 0.0)
    sigma0_sq = tf.maximum(tf.cast(no_bg, tf.float32), eps)
    sigma1_sq = sigma0_sq * (1.0 + gamma_tf)

    if sigma0_sq.shape.rank is not None and sigma0_sq.shape.rank > 0:
        sigma0_sq = tf.expand_dims(sigma0_sq, axis=-1)
        sigma1_sq = tf.expand_dims(sigma1_sq, axis=-1)

    pts = tf.cast(c_points, tf.complex64)
    diff = tf.expand_dims(tf.cast(y, tf.complex64), axis=-1) - pts
    dist_sq = tf.math.square(tf.math.real(diff)) + tf.math.square(tf.math.imag(diff))

    log_c0 = tf.math.log(1.0 - p_tf) - tf.math.log(sigma0_sq) - dist_sq / sigma0_sq
    log_c1 = tf.math.log(p_tf) - tf.math.log(sigma1_sq) - dist_sq / sigma1_sq
    log_p_y_s = tf.math.reduce_logsumexp(tf.stack([log_c0, log_c1], axis=-1), axis=-1)

    log_p_exp = tf.expand_dims(log_p_y_s, axis=-2)
    neg_inf = tf.constant(-np.inf, dtype=tf.float32)
    log_p_1 = tf.where(masks_1, log_p_exp, neg_inf)
    log_p_0 = tf.where(masks_0, log_p_exp, neg_inf)
    llr_sym = tf.math.reduce_logsumexp(log_p_1, axis=-1) - tf.math.reduce_logsumexp(log_p_0, axis=-1)
    llr_sym = tf.where(tf.math.is_finite(llr_sym), llr_sym, tf.zeros_like(llr_sym))

    num_bits_per_sym = tf.shape(masks_1)[0]
    out_shape = tf.concat([tf.shape(y)[:-1], [tf.shape(y)[-1] * num_bits_per_sym]], axis=0)
    return tf.reshape(llr_sym, out_shape)


def apply_channel_and_equalizer(x, ebno_db, coderate, params=None):
    """Aplica canal, error CSI, ecualización, demapeo con recorte LLR y desintercalado."""
    p = get_sim_params() if params is None else params
    y, h, no = sample_active_channel(x, ebno_db, coderate, p)

    csi_err_eff = float(p["csi_err_var"]) if p["canal"] not in CANALES_SIN_FADING else 0.0
    h_obs = (
        aplicar_error_csi(h, p["bloq_coher"], tf.constant(csi_err_eff, tf.float32))
        if csi_err_eff > 0.0 else h
    )

    eq = p["ecualizador"]
    if eq == "no_eq":
        x_hat, no_eff = sin_ecualizar(y, h_obs, no)
    elif eq == "zf":
        x_hat, no_eff = zf(y, h_obs, no)
    elif eq == "mmse":
        x_hat, no_eff = mmse(y, h_obs, no, csi_err_var=tf.constant(csi_err_eff, tf.float32))
    else:
        raise ValueError(f"Ecualizador inválido: {eq}")

    if p["canal"] == "impulsive" and p["imp_demapper"] == "mixture":
        p_tf = tf.cast(p["imp_prob"], tf.float32)
        gamma_tf = tf.cast(p["imp_gamma"], tf.float32)
        no_bg = tf.cast(no_eff, tf.float32) / (1.0 + p_tf * gamma_tf)
        raw_llr = demap_impulsive_mixture(
            x_hat, constellation_points, bit_masks_1, bit_masks_0, no_bg, p_tf, gamma_tf
        )
    else:
        raw_llr = demapeador(x_hat, no_eff)

    llr_clip = float(p["LLR_clip"])
    llr_prev = tf.clip_by_value(raw_llr, -llr_clip, llr_clip)
    return deinterleaver(llr_prev)

---
## Módulo 4: Estimación Automática del SNR de Diseño vía Información Mutua

La construcción de un código Polar requiere seleccionar un **SNR de diseño** $(E_b/N_0)_{\text{diseño}}$ apropiado:
- Si el SNR de diseño es **demasiado bajo**, la mayoría de bit-canales se mantienen ruidosos y el ordenamiento pierde precisión.
- Si es **demasiado alto**, más de $K$ bit-canales registran $0$ errores empíricos ($\widehat{\text{BER}}_i = 0$) dentro del presupuesto Monte Carlo $M$, saturando la resolución del ranking.

### Estimador Empírico de Información Mutua a Nivel de Bit (BICM / GMI)
Para bits equiprobables $b \in \{0, 1\}$ y LLRs en convención Sionna $L = \ln\frac{P(b=1\mid y)}{P(b=0\mid y)}$, la información mutua por bit (o *Generalized Mutual Information* bajo demapeo desajustado) se estima sin sesgo como:
$$I(W) = 1 - \mathbb{E}\!\left[\log_2\!\left(1 + e^{-(2b - 1)L}\right)\right] = 1 - \frac{1}{\ln 2}\,\mathbb{E}\!\left[\operatorname{softplus}\!\left(-(2b-1)L\right)\right]$$
Buscamos por bisección el $E_b/N_0$ donde $I(W) \approx \alpha \cdot R$ (por defecto $\alpha = 0.88$, situando el punto de operación ligeramente debajo del límite de capacidad del canal de tasa $R = K/N$).

### Dos Refinamientos Numéricos Clave
1. **Common Random Numbers (CRN):** Restablecer `sionna.phy.config.seed` antes de evaluar cada punto medio de la bisección garantiza que $\hat{I}(\text{SNR})$ sea monótona sobre la misma realización muestral, evitando oscilaciones espurias del algoritmo de bisección.
2. **Escalamiento adaptativo de bloques de desvanecimiento:** En canales lentos o planos (`bloq_coher = N`), cada trama aporta pocos bloques independientes de $h$; el número de lotes se escala automáticamente para acumular al menos $4096$ bloques de coherencia por evaluación.

In [ ]:
# =====================================================================
# ESTIMACIÓN AUTOMÁTICA DEL SNR DE DISEÑO (BISECCIÓN DE I(W) CON CRN)
# =====================================================================

AUTO_SNR_TARGET_ALPHA = 0.88   # Objetivo: I(W) ≈ alpha * R
AUTO_SNR_BISECTION_STEPS = 12  # Precisión ≈ 60 / 2^12 ≈ 0.015 dB en [-20, 40] dB
APLICAR_SNR_AUTO = True        # True: actualiza snr_simulacion automáticamente

p_auto = get_sim_params()
coderate_auto = float(p_auto["K"] / p_auto["N"])
target_mi = float(np.clip(AUTO_SNR_TARGET_ALPHA * coderate_auto, 0.02, 0.98))

# Escala num_batches según la diversidad de desvanecimiento para estabilizar la varianza
batch_size_auto = max(64, min(2048, 65536 // p_auto["N"]))
if p_auto["canal"] in CANALES_SIN_FADING:
    num_batches_auto = 4
else:
    n_sym_auto = max(1, p_auto["N"] // p_auto["bitsxsimb"])
    bloq_c_auto = max(1, int(p_auto["bloq_coher"]))
    blocks_per_frame = max(1, (n_sym_auto + bloq_c_auto - 1) // bloq_c_auto)
    fading_per_batch = batch_size_auto * blocks_per_frame
    num_batches_auto = max(4, min(64, (4096 + fading_per_batch - 1) // fading_per_batch))


@tf.function
def _eval_mi_batch(ebno_db_tf):
    b = datos([batch_size_auto, p_auto["N"]])
    x = mapeador(interleaver(b))
    llr = apply_channel_and_equalizer(x, ebno_db_tf, coderate=coderate_auto, params=p_auto)
    aligned_llr = (2.0 * tf.cast(b, tf.float32) - 1.0) * tf.cast(llr, tf.float32)
    mi = 1.0 - tf.reduce_mean(tf.math.softplus(-aligned_llr) / tf.math.log(2.0))
    b_hat = tf.cast(llr > 0.0, b.dtype)
    ber = tf.reduce_mean(tf.cast(tf.not_equal(b, b_hat), tf.float32))
    return mi, ber


def _eval_snr_crn(snr_db_val):
    sionna.phy.config.seed = p_auto["seed"]  # CRN: misma trayectoria muestral
    ebno_tf = tf.constant(float(snr_db_val), dtype=tf.float32)
    mi_sum, ber_sum = 0.0, 0.0
    for _ in range(num_batches_auto):
        mi_t, ber_t = _eval_mi_batch(ebno_tf)
        mi_sum += float(mi_t.numpy())
        ber_sum += float(ber_t.numpy())
    return float(np.clip(mi_sum / num_batches_auto, 0.0, 1.0)), ber_sum / num_batches_auto


lo, hi = -20.0, 40.0
mi_lo, ber_lo = _eval_snr_crn(lo)
mi_hi, ber_hi = _eval_snr_crn(hi)

print(
    f"⚡ Bisección Auto-SNR | Canal={p_auto['canal']}, Eq={p_auto['ecualizador']} (sigma_e^2={p_auto['csi_err_var']}), "
    f"R={coderate_auto:.3f}, I(W) objetivo={target_mi:.4f} | lotes={num_batches_auto}×{batch_size_auto}"
)

hist_snr, hist_mi = [lo, hi], [mi_lo, mi_hi]
if mi_lo >= target_mi:
    print(f"⚠️ Raíz por debajo de {lo:.1f} dB (I(W)={mi_lo:.4f} >= {target_mi:.4f}).")
    best_snr, best_mi, best_ber = lo, mi_lo, ber_lo
elif mi_hi <= target_mi:
    print(f"⚠️ Raíz por encima de {hi:.1f} dB o inalcanzable (I(W)={mi_hi:.4f} <= {target_mi:.4f}).")
    best_snr, best_mi, best_ber = hi, mi_hi, ber_hi
else:
    best_snr, best_mi, best_ber = 0.0, 0.0, 0.5
    for step in range(1, AUTO_SNR_BISECTION_STEPS + 1):
        mid = 0.5 * (lo + hi)
        mi_val, ber_val = _eval_snr_crn(mid)
        best_snr, best_mi, best_ber = mid, mi_val, ber_val
        hist_snr.append(mid)
        hist_mi.append(mi_val)
        print(f"  Paso {step:2d}/{AUTO_SNR_BISECTION_STEPS} | Eb/N0 = {mid:+6.2f} dB | I(W) = {mi_val:.4f}")
        if mi_val < target_mi:
            lo = mid
        else:
            hi = mid
    if abs(best_mi - target_mi) > 0.03:
        print(f"⚠️ Advertencia: I(W) convergido ({best_mi:.4f}) difiere > 0.03 del objetivo ({target_mi:.4f}).")

snr_auto_db = round(best_snr, 2)
print(
    f"\n✅ SNR de diseño recomendado: {snr_auto_db:.2f} dB "
    f"(I(W) = {best_mi:.4f} / objetivo {target_mi:.4f}, BER sin código ≈ {best_ber:.3e})"
)

if APLICAR_SNR_AUTO:
    snr_simulacion = snr_auto_db
    print(f"   → snr_simulacion actualizado a {snr_simulacion:.2f} dB.")
else:
    print(f"   → snr_simulacion conservado en {snr_simulacion:.2f} dB (APLICAR_SNR_AUTO=False).")

# Visualización compacta de la curva I(W) vs Eb/N0 evaluada por la bisección CRN
order_mi = np.argsort(hist_snr)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(np.asarray(hist_snr)[order_mi], np.asarray(hist_mi)[order_mi], "o-", color="#2b6cb0", lw=1.8, ms=5, label="I(W) evaluada (CRN)")
ax.axhline(target_mi, color="#2f855a", ls="--", lw=1.4, label=f"Objetivo α·R = {target_mi:.3f}")
ax.axvline(snr_auto_db, color="#c53030", ls=":", lw=1.5, label=f"SNR diseño = {snr_auto_db:.2f} dB")
ax.set_title(f"Bisección de Información Mutua — Canal {p_auto['canal']} (R={coderate_auto:.3f})")
ax.set_xlabel(r"$E_b/N_0$ [dB]")
ax.set_ylabel("Información Mutua I(W) [bits/uso]")
ax.set_ylim(-0.03, 1.03)
ax.legend()
plt.tight_layout()
plt.show()

# Limpia grafo temporal y restaura semilla limpia para etapas posteriores
del _eval_mi_batch
gc.collect()
sionna.phy.config.seed = SEED

---
## Módulo 5: Polarización de Canal y Construcción Monte Carlo (Genie-Aided SC)

### 5.1 Transformada Polar de Arıkan y Bit-Canales Sintéticos
Dado un vector binario $\mathbf{u} \in \{0, 1\}^N$ ($N = 2^n$), la codificación polar aplica la mariposa XOR de Arıkan:
$$\mathbf{x} = \mathbf{u} \cdot G_N \pmod 2, \qquad G_N = F^{\otimes n}, \qquad F = \begin{bmatrix} 1 & 0 \\ 1 & 1 \end{bmatrix}$$
Al transmitir $\mathbf{x}$ sobre $N$ usos del canal físico $W$ y decodificar secuencialmente $u_0, u_1, \dots, u_{N-1}$, cada bit $u_i$ percibe un **bit-canal sintético** $W_N^{(i)}(y_0^{N-1}, u_0^{i-1} \mid u_i)$. El teorema de polarización establece que cuando $N \to \infty$, una fracción $I(W)$ de los bit-canales converge a capacidad $1$ (sin ruido) y la fracción restante $1 - I(W)$ converge a capacidad $0$ (puro ruido).

### 5.2 Dos Motores Seleccionables (`mc_modo`)

Este cuaderno ofrece dos modos seleccionables desde el Módulo 1 mediante la variable `mc_modo`:

1. **Modo `"genie_sc"` (vectorizado en una sola pasada — recomendado, $\mathcal{O}(N \log N)$ por trama):**
   Aprovecha la definición exacta de $W_N^{(i)}(y_0^{N-1}, u_0^{i-1} \mid u_i)$: el decodificador conoce los bits previos verdaderos $u_0^{i-1}$ (**asistencia por genio**). Precalculando las sumas parciales verdaderas en las $n = \log_2 N$ etapas de la mariposa, se propagan los LLRs en una única pasada vectorizada. En la convención de Arıkan ($L = -L_{\text{Sionna}}$), cada etapa combina pares $(a, b)$ mediante:
   - **Rama superior (box-plus exacto con corrección de Jacobi, libre de desbordamiento):**
     $$L^- = a \boxplus b = \operatorname{sign}(a)\operatorname{sign}(b)\min(|a|, |b|) + \operatorname{softplus}(-|a+b|) - \operatorname{softplus}(-|a-b|)$$
   - **Rama inferior (condicionada a la suma parcial verdadera $u_{\text{left}} \in \{0, 1\}$ provista por el genio):**
     $$L^+ = (1 - 2 u_{\text{left}})\, a + b$$
   Al final de las $n$ etapas se deciden simultáneamente todos los $\hat{u}_i = \mathbb{1}[L_i < 0]$. Al promediar sobre vectores $\mathbf{u}$ equiprobables aleatorios, este método es exacto incluso en modulaciones/canales asimétricos (e.g., 16-QAM).

2. **Modo `"por_bit_canal"` (iterativo por cada bit-canal $i$ — referencia didáctica, $\mathcal{O}(N^2 \log N)$):**
   Itera explícitamente sobre cada posición $i \in \{0, 1, \dots, N-1\}$ congelando todos los demás bits ($\mathcal{F}_i = \{j \neq i\}$) y transmitiendo únicamente 1 bit de información en la posición $i$ mediante `PolarEncoder` y `PolarSCLDecoder(list_size=1)` de Sionna. Es computacionalmente más lento ($\sim N\times$), pero resulta ideal para verificar directamente la equivalencia empírica contra el decodificador SC nativo de Sionna en canales simétricos (BPSK) y medir el factor de aceleración del motor vectorizado.

> **Nota teórica:** El modo `"por_bit_canal"` fija los bits congelados $u_{j \neq i} = 0$, lo cual es rigurosamente equivalente a promediar sobre $u_0^{i-1}$ en canales simétricos (e.g., BPSK), mientras que `"genie_sc"` sortea $\mathbf{u}$ completo uniformemente y generaliza sin sesgo a constelaciones de orden superior.

### 5.3 Desempate Determinístico por *Polarization Weight* y Cota de Unión SC
En simulaciones finitas, múltiples bit-canales muy confiables pueden empatar en $\widehat{\text{BER}}_i = 0$. Ordenar con `np.argsort` rompería esos empates arbitrariamente por índice. Para resolver empates de forma teóricamente consistente con el orden parcial universal (alineado con 3GPP 5G NR), desempatamos por mayor **Polarization Weight** ($\beta = 2^{1/4}$):
$$W_{\text{PW}}(i) = \sum_{k=0}^{n-1} b_k(i)\, \beta^{k}, \qquad i = \sum_{k=0}^{n-1} b_k(i)\, 2^k, \quad \beta = 2^{1/4}$$
Además, el perfil $\widehat{\text{BER}}_i$ entrega directamente la **cota de unión de Arıkan sobre la BLER del decodificador SC** en el SNR de diseño:
$$\text{BLER}_{\text{SC}} \le \sum_{i \in \mathcal{A}} P_e\!\left(W_N^{(i)}\right)$$

In [ ]:
# =====================================================================
# PRIMITIVAS POLAR XLA: POLARIZATION WEIGHT, MARIPOSA Y GENIE-AIDED SC
# =====================================================================

def compute_polarization_weights(n_len):
    """Calcula el vector de pesos de polarización W_PW(i) con beta = 2^(1/4) (3GPP)."""
    n_bits = int(np.log2(n_len))
    indices = np.arange(n_len, dtype=np.int64)
    k = np.arange(n_bits, dtype=np.int64)[:, None]
    bits = (indices[None, :] >> k) & 1
    beta_powers = np.power(2.0, np.arange(n_bits, dtype=np.float64) / 4.0)[:, None]
    return np.sum(bits * beta_powers, axis=0)


def rank_polar_bit_channels(ber_bits):
    """Ordena bit-canales por BER ascendente desempatando por mayor W_PW(i)."""
    ber_arr = np.asarray(ber_bits, dtype=np.float64)
    pw_weights = compute_polarization_weights(len(ber_arr))
    return np.lexsort((-pw_weights, ber_arr))


def _boxplus_exact(a, b):
    """Operador box-plus exacto en dominio logarítmico (corrección de Jacobi)."""
    min_sum = tf.sign(a) * tf.sign(b) * tf.minimum(tf.abs(a), tf.abs(b))
    jacobi = tf.math.softplus(-tf.abs(a + b)) - tf.math.softplus(-tf.abs(a - b))
    return min_sum + jacobi


@tf.function(jit_compile=True)
def polar_encode_butterfly(u_bits):
    """Codifica u_bits [batch, N] -> x_bits = u * G_N mod 2 mediante mariposa XOR O(N log N)."""
    x = tf.cast(u_bits, tf.uint8)
    batch_size = tf.shape(x)[0]
    n_len = u_bits.shape[-1]
    stride = 1
    while stride < n_len:
        x_blk = tf.reshape(x, [batch_size, n_len // (2 * stride), 2, stride])
        left = tf.bitwise.bitwise_xor(x_blk[:, :, 0, :], x_blk[:, :, 1, :])
        right = x_blk[:, :, 1, :]
        x = tf.reshape(tf.stack([left, right], axis=2), [batch_size, n_len])
        stride *= 2
    return tf.cast(x, tf.float32)


@tf.function(jit_compile=True)
def polar_genie_sc_errors(llr_sionna, u_bits):
    """Motor SC vectorizado asistido por genio en una sola pasada (retorna errores [N] por lote)."""
    u_true = tf.cast(u_bits, tf.uint8)
    batch_size = tf.shape(u_true)[0]
    n_len = u_bits.shape[-1]

    # Precalcula sumas parciales verdaderas (el genio) en cada etapa de la mariposa
    stages = [u_true]
    cur = u_true
    stride = 1
    while stride < n_len:
        cur_blk = tf.reshape(cur, [batch_size, n_len // (2 * stride), 2, stride])
        left = tf.bitwise.bitwise_xor(cur_blk[:, :, 0, :], cur_blk[:, :, 1, :])
        right = cur_blk[:, :, 1, :]
        cur = tf.reshape(tf.stack([left, right], axis=2), [batch_size, n_len])
        stages.append(cur)
        stride *= 2

    # Convierte LLR de Sionna ln(P(1)/P(0)) a convención de Arıkan ln(P(0)/P(1))
    llr = -tf.cast(llr_sionna, tf.float32)
    num_stages = len(stages) - 1
    for stage_idx in range(num_stages - 1, -1, -1):
        stage_stride = 1 << stage_idx
        num_blocks = n_len // (2 * stage_stride)
        llr_blk = tf.reshape(llr, [batch_size, num_blocks, 2, stage_stride])
        a = llr_blk[:, :, 0, :]
        b = llr_blk[:, :, 1, :]
        u_left = tf.cast(
            tf.reshape(stages[stage_idx], [batch_size, num_blocks, 2, stage_stride])[:, :, 0, :],
            tf.float32,
        )
        llr_upper = _boxplus_exact(a, b)
        llr_lower = (1.0 - 2.0 * u_left) * a + b
        llr = tf.reshape(tf.stack([llr_upper, llr_lower], axis=2), [batch_size, n_len])

    u_hat = tf.cast(llr < 0.0, tf.uint8)
    return tf.math.count_nonzero(tf.not_equal(u_hat, u_true), axis=0, dtype=tf.int64)

In [ ]:
# =====================================================================
# GENERADOR MONTE CARLO SELECCIONABLE (GENIE-AIDED SC O POR BIT-CANAL)
# =====================================================================

REANUDAR_BACKUP = True       # False: descarta cualquier respaldo previo y empieza de cero
MC_BACKUP_INTERVAL = 20      # En genie_sc: respaldo atómico cada N iteraciones
MC_BIT_BACKUP_INTERVAL = 10  # En por_bit_canal: respaldo atómico cada N bit-canales
MC_CHUNK_SYNC = 50           # Sincronización GPU->CPU por bloque en modo por_bit_canal
MC_RAM_LIMIT_PERCENT = 85.0  # Umbral de protección preventiva de RAM
MC_RESUME_SEED_STRIDE = 1_000_003

p_mc = get_sim_params()
modo_activo = p_mc["mc_modo"]  # "genie_sc" o "por_bit_canal"
backup_file = f"BER_bits_backup_{p_mc['N']}.npz"
MC_META_KEYS = (
    "N", "K", "bitsxsimb", "canal", "bloq_coher", "k_factor_db", "sigma_db",
    "m_nakagami", "imp_prob", "imp_gamma", "imp_demapper", "ecualizador",
    "csi_err_var", "LLR_clip", "snr_simulacion", "seed", "lote_mc", "num_iter",
)
mc_meta_actual = {k: p_mc[k] for k in MC_META_KEYS}


def guardar_backup_mc(ruta, ber_arr, next_index, meta, error_counts=None, completed_iters=None):
    """Guarda respaldo atómico .npz compatible con gui.py (vectorizado o por bit-canal)."""
    tmp_path = f"{ruta}.tmp.npz"
    payload = {
        "ber_bits": np.asarray(ber_arr, dtype=np.float64),
        "next_index": np.int64(next_index),
        "meta_json": np.array(json.dumps(meta, sort_keys=True)),
    }
    if error_counts is not None and completed_iters is not None:
        payload["error_counts"] = np.asarray(error_counts, dtype=np.int64)
        payload["completed_iters"] = np.int64(completed_iters)
    np.savez(tmp_path, **payload)
    os.replace(tmp_path, ruta)


def cargar_backup_mc(ruta):
    """Carga un respaldo .npz devolviendo (ber_bits, next_index, error_counts, completed_iters, meta)."""
    with np.load(ruta, allow_pickle=False) as data:
        ber_arr = np.array(data["ber_bits"], dtype=np.float64)
        next_idx = int(data["next_index"])
        meta = json.loads(str(data["meta_json"]))
        err_counts = np.array(data["error_counts"], dtype=np.int64) if "error_counts" in data.files else None
        comp_iters = int(data["completed_iters"]) if "completed_iters" in data.files else None
    return ber_arr, next_idx, err_counts, comp_iters, meta


def formatear_duracion(segundos):
    segundos = max(0, int(round(segundos)))
    h, rem = divmod(segundos, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h > 0 else f"{m:02d}:{s:02d}"


def _ejecutar_mc_genie_sc(p_cfg, b_file, meta_actual):
    """Motor vectorizado Genie-Aided SC en una sola pasada sobre los N bit-canales."""
    n_len = int(p_cfg["N"])
    lote_mc_int = int(p_cfg["lote_mc"])
    num_iter_int = int(p_cfg["num_iter"])
    error_counts = np.zeros(n_len, dtype=np.int64)
    completed_iters = 0

    if REANUDAR_BACKUP and os.path.exists(b_file):
        try:
            _, _, saved_counts, saved_iters, saved_meta = cargar_backup_mc(b_file)
            if (
                saved_counts is not None
                and saved_iters is not None
                and len(saved_counts) == n_len
                and saved_meta == meta_actual
                and 0 <= saved_iters <= num_iter_int
            ):
                error_counts = saved_counts
                completed_iters = saved_iters
                print(f"🔄 Backup vectorizado recuperado ({b_file}). Retomando desde iteración {completed_iters}/{num_iter_int}.")
            else:
                print(f"⚠️ El backup '{b_file}' no coincide con la configuración/modo actual; se iniciará desde cero.")
                os.remove(b_file)
        except Exception as exc:
            print(f"⚠️ No se pudo leer el backup '{b_file}' ({exc}); se iniciará desde cero.")
    elif not REANUDAR_BACKUP and os.path.exists(b_file):
        os.remove(b_file)

    # Desplaza semilla al reanudar para evitar repetir los mismos lotes aleatorios
    sionna.phy.config.seed = int(p_cfg["seed"]) + completed_iters * MC_RESUME_SEED_STRIDE

    snr_sim_tf = tf.constant(p_cfg["snr_simulacion"], dtype=tf.float32)
    coderate_sim = float(p_cfg["K"] / n_len)
    log_interval = max(MC_BACKUP_INTERVAL, num_iter_int // 10)

    @tf.function
    def _simulacion_genie_batch():
        u_bits = datos([lote_mc_int, n_len])
        x_bits = polar_encode_butterfly(u_bits)
        x = mapeador(interleaver(x_bits))
        llr = apply_channel_and_equalizer(x, snr_sim_tf, coderate=coderate_sim, params=p_cfg)
        return polar_genie_sc_errors(llr, u_bits)

    print(
        f"Modo 'genie_sc' (vectorizado O(N log N)) | Canal={p_cfg['canal']}, Eq={p_cfg['ecualizador']}, "
        f"Eb/N0={p_cfg['snr_simulacion']:.2f} dB | {num_iter_int} × {lote_mc_int} = {num_iter_int * lote_mc_int:,} muestras/bit-canal"
    )

    inicio = time.monotonic()
    start_iter = completed_iters
    detenido = False

    try:
        while completed_iters < num_iter_int:
            paso = min(MC_BACKUP_INTERVAL, num_iter_int - completed_iters)
            err_chunk = tf.zeros([n_len], dtype=tf.int64)
            for _ in range(paso):
                err_chunk = err_chunk + _simulacion_genie_batch()
            error_counts += err_chunk.numpy()
            completed_iters += paso

            ber_est = error_counts / float(completed_iters * lote_mc_int)
            guardar_backup_mc(
                b_file, ber_est, completed_iters, meta_actual,
                error_counts=error_counts, completed_iters=completed_iters,
            )

            elapsed = time.monotonic() - inicio
            session_done = completed_iters - start_iter
            eta = (elapsed / session_done) * (num_iter_int - completed_iters) if session_done > 0 else 0.0
            ram_pct = psutil.virtual_memory().percent

            if completed_iters == num_iter_int or completed_iters == paso or completed_iters % log_interval < MC_BACKUP_INTERVAL:
                print(
                    f"  Iter {completed_iters:4d}/{num_iter_int} | BER medio={np.mean(ber_est):.3e} | "
                    f"mín/máx={np.min(ber_est):.2e}/{np.max(ber_est):.2e} | "
                    f"RAM={ram_pct:.0f}% | ETA={formatear_duracion(eta)}"
                )

            if ram_pct > MC_RAM_LIMIT_PERCENT:
                print(f"\n🚨 RAM al {ram_pct:.1f}% (umbral {MC_RAM_LIMIT_PERCENT:.0f}%). Respaldo guardado en iteración {completed_iters}/{num_iter_int}.")
                break
    except KeyboardInterrupt:
        detenido = True
        if completed_iters > 0:
            ber_parcial = error_counts / float(completed_iters * lote_mc_int)
            guardar_backup_mc(
                b_file, ber_parcial, completed_iters, meta_actual,
                error_counts=error_counts, completed_iters=completed_iters,
            )
        print(f"\n⚠️ Interrupción manual en iteración {completed_iters}/{num_iter_int}." + (" Respaldo guardado." if completed_iters > 0 else ""))
    finally:
        del _simulacion_genie_batch
        gc.collect()

    if completed_iters == num_iter_int:
        return error_counts / float(num_iter_int * lote_mc_int), time.monotonic() - inicio, False
    return None, time.monotonic() - inicio, detenido


def _ejecutar_mc_por_bit_canal(p_cfg, b_file, meta_actual):
    """Modo clásico por bit-canal: itera i = 0..N-1 congelando todos los bits excepto i."""
    n_len = int(p_cfg["N"])
    lote_mc_int = int(p_cfg["lote_mc"])
    num_iter_int = int(p_cfg["num_iter"])
    ber_arr = np.full(n_len, np.nan, dtype=np.float64)
    start_index = 0

    if REANUDAR_BACKUP and os.path.exists(b_file):
        try:
            saved_ber, saved_idx, saved_counts, _, saved_meta = cargar_backup_mc(b_file)
            if (
                saved_counts is None
                and len(saved_ber) == n_len
                and saved_meta == meta_actual
                and 0 <= saved_idx <= n_len
            ):
                ber_arr = saved_ber
                start_index = saved_idx
                print(f"🔄 Backup por bit-canal recuperado ({b_file}). Retomando desde bit-canal {start_index + 1}/{n_len}.")
            else:
                print(f"⚠️ El backup '{b_file}' no coincide con la configuración/modo actual; se iniciará desde cero.")
                os.remove(b_file)
        except Exception as exc:
            print(f"⚠️ No se pudo leer el backup '{b_file}' ({exc}); se iniciará desde cero.")
    elif not REANUDAR_BACKUP and os.path.exists(b_file):
        os.remove(b_file)

    sionna.phy.config.seed = int(p_cfg["seed"]) + start_index * MC_RESUME_SEED_STRIDE

    snr_sim_tf = tf.constant(p_cfg["snr_simulacion"], dtype=tf.float32)
    coderate_sim = float(p_cfg["K"] / n_len)
    bits_totales = lote_mc_int * num_iter_int
    ultimo_completado = start_index

    print(
        f"Modo 'por_bit_canal' (iterativo O(N^2 log N)) | Canal={p_cfg['canal']}, Eq={p_cfg['ecualizador']}, "
        f"Eb/N0={p_cfg['snr_simulacion']:.2f} dB | {n_len} bit-canales × {bits_totales:,} muestras"
    )

    inicio = time.monotonic()
    detenido = False

    try:
        for i in range(start_index, n_len):
            bits_cong_i = np.array([j for j in range(n_len) if j != i])
            codif_i = sionna.phy.fec.polar.encoding.PolarEncoder(bits_cong_i, n_len)
            decodif_i = sionna.phy.fec.polar.decoding.PolarSCLDecoder(bits_cong_i, n_len, list_size=1)

            # Grafo TF estándar (sin jit_compile por bit-canal para evitar fugas de caché XLA)
            @tf.function
            def _sim_bit_batch():
                bits = datos([lote_mc_int, 1])
                x = mapeador(interleaver(codif_i(bits)))
                llr = apply_channel_and_equalizer(x, snr_sim_tf, coderate=coderate_sim, params=p_cfg)
                bits_dec = decodif_i(llr)
                err = tf.math.count_nonzero(tf.not_equal(bits, bits_dec), dtype=tf.int32)
                return tf.cast(err, tf.int64)

            try:
                errores_tot = 0
                iter_rest = num_iter_int
                while iter_rest > 0:
                    paso = min(MC_CHUNK_SYNC, iter_rest)
                    err_blk = tf.constant(0, dtype=tf.int64)
                    for _ in range(paso):
                        err_blk = err_blk + _sim_bit_batch()
                    errores_tot += int(err_blk.numpy())
                    iter_rest -= paso
            finally:
                del _sim_bit_batch, codif_i, decodif_i
                gc.collect()

            ber_arr[i] = errores_tot / float(bits_totales)
            ultimo_completado = i + 1

            elapsed = time.monotonic() - inicio
            session_done = ultimo_completado - start_index
            eta = (elapsed / session_done) * (n_len - ultimo_completado) if session_done > 0 else 0.0
            ram_pct = psutil.virtual_memory().percent
            print(
                f"  Bit-canal {ultimo_completado:4d}/{n_len} | BER={ber_arr[i]:.2e} | "
                f"RAM={ram_pct:.0f}% | ETA={formatear_duracion(eta)}"
            )

            if ultimo_completado % MC_BIT_BACKUP_INTERVAL == 0:
                guardar_backup_mc(b_file, ber_arr, ultimo_completado, meta_actual)

            if ram_pct > MC_RAM_LIMIT_PERCENT:
                guardar_backup_mc(b_file, ber_arr, ultimo_completado, meta_actual)
                print(f"\n🚨 RAM al {ram_pct:.1f}% (umbral {MC_RAM_LIMIT_PERCENT:.0f}%). Respaldo guardado en bit-canal {ultimo_completado}/{n_len}.")
                break
    except KeyboardInterrupt:
        detenido = True
        if ultimo_completado > 0:
            guardar_backup_mc(b_file, ber_arr, ultimo_completado, meta_actual)
        print(f"\n⚠️ Interrupción manual en bit-canal {ultimo_completado}/{n_len}." + (" Respaldo guardado." if ultimo_completado > 0 else ""))

    if ultimo_completado == n_len and np.all(np.isfinite(ber_arr)):
        return ber_arr, time.monotonic() - inicio, False
    return None, time.monotonic() - inicio, detenido


if modo_activo == "genie_sc":
    ber_res, duracion_mc, detenido_manual = _ejecutar_mc_genie_sc(p_mc, backup_file, mc_meta_actual)
elif modo_activo == "por_bit_canal":
    ber_res, duracion_mc, detenido_manual = _ejecutar_mc_por_bit_canal(p_mc, backup_file, mc_meta_actual)
else:
    raise ValueError(f"mc_modo no reconocido: {modo_activo}")

if ber_res is not None:
    BER_bits = ber_res
    zero_ber_count = int(np.count_nonzero(BER_bits == 0.0))
    if zero_ber_count > p_mc["K"]:
        print(
            f"⚠️ Advertencia: {zero_ber_count}/{p_mc['N']} bit-canales registraron BER = 0.0 (> K={p_mc['K']}). "
            "Considere reducir snr_simulacion; empates resueltos con Polarization Weight (beta = 2^(1/4))."
        )
    indices_ordenados = rank_polar_bit_channels(BER_bits)
    bits_info = indices_ordenados[:p_mc["K"]]
    bits_congelados = indices_ordenados[-(p_mc["N"] - p_mc["K"]):]
    if os.path.exists(backup_file):
        os.remove(backup_file)
    print(f"✅ Construcción Monte Carlo ({modo_activo}) completada al 100% en {formatear_duracion(duracion_mc)}.")
elif not detenido_manual:
    print("⚠️ La simulación no finalizó completamente; ejecute esta celda nuevamente para reanudar.")

# Restaura la semilla base para que el Módulo 7 sea reproducible independientemente del modo MC
sionna.phy.config.seed = SEED

In [ ]:
# =====================================================================
# DIAGNÓSTICO VISUAL DE POLARIZACIÓN, COTA DE UNIÓN SC Y COMPARACIÓN PW
# =====================================================================

assert "BER_bits" in globals() and len(BER_bits) == N and np.all(np.isfinite(BER_bits)), (
    "Ejecute el generador Monte Carlo (o cargue un CSV válido) antes de ejecutar este diagnóstico."
)

indices_mc = rank_polar_bit_channels(BER_bits)
info_mc = set(indices_mc[:K].tolist())

# Comparación frente al orden universal 3GPP Polarization Weight (independiente del canal)
pw_weights = compute_polarization_weights(N)
indices_pw = np.argsort(-pw_weights)
info_pw = set(indices_pw[:K].tolist())

coincidencias = len(info_mc & info_pw)
pct_coincidencia = 100.0 * coincidencias / K
bits_intercambiados = K - coincidencias

# Métricas de polarización y cota de unión SC en el SNR de diseño
union_sum_sc = float(np.sum(BER_bits[indices_mc[:K]]))
union_bound_sc = min(1.0, union_sum_sc)
frac_muy_buenos = 100.0 * np.mean(BER_bits < 1e-3)
frac_muy_malos = 100.0 * np.mean(BER_bits > 0.4)
frac_transicion = 100.0 - frac_muy_buenos - frac_muy_malos
zero_count = int(np.count_nonzero(BER_bits == 0.0))

print(f"📈 Análisis de Polarización (N={N}, K={K}, Canal={canal}, SNR_diseño={snr_simulacion:.2f} dB):")
print(f"   • Canales casi sin ruido (BER < 1e-3): {frac_muy_buenos:5.1f}%  |  Empates BER = 0: {zero_count}/{N}")
print(f"   • Canales muy ruidosos   (BER > 0.40): {frac_muy_malos:5.1f}%  |  Zona de transición: {frac_transicion:5.1f}%")
print(
    f"   • Cota de unión SC BLER en SNR de diseño: min(1, sum_{{i in A}} BER_i) = {union_bound_sc:.4e} "
    f"(suma cruda = {union_sum_sc:.4e})"
)
print(
    f"   • Coincidencia conjunto información MC vs. PW (beta=2^(1/4)): "
    f"{coincidencias}/{K} ({pct_coincidencia:.1f}%, {bits_intercambiados} bit-canales reasignados por el canal)"
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
idx_nat = np.arange(N)
is_info = np.isin(idx_nat, list(info_mc))

# Panel izquierdo: estructura fractal de BER_i en orden natural
axes[0].scatter(idx_nat[~is_info], BER_bits[~is_info], s=10, color="#c53030", alpha=0.75, label=f"Congelados F ({N-K})")
axes[0].scatter(idx_nat[is_info], BER_bits[is_info], s=10, color="#2b6cb0", alpha=0.85, label=f"Información A ({K})")
axes[0].set_title("Estructura de Polarización por Índice Natural i")
axes[0].set_xlabel("Índice de bit-canal i")
axes[0].set_ylabel("BER empírico de bit-canal")
axes[0].legend()

# Panel derecho: curva ordenada en escala logarítmica (transición de polarización)
ber_sorted = BER_bits[indices_mc]
pos_ber = ber_sorted[ber_sorted > 0.0]
ber_floor = 0.5 / (lote_mc * num_iter)
if len(pos_ber) > 0:
    ber_floor = min(ber_floor, float(np.min(pos_ber)) * 0.5)
axes[1].semilogy(
    np.arange(1, N + 1),
    np.maximum(ber_sorted, ber_floor),
    color="#1a365d",
    lw=2,
    label=f"BER ordenada (piso para BER=0: {ber_floor:.1e})",
)
axes[1].axvline(K, color="#2f855a", ls="--", lw=1.8, label=f"Corte K = {K} (R = {K/N:.3f})")
axes[1].set_title("Perfil Ordenado de Confiabilidad (Transición de Polarización)")
axes[1].set_xlabel("Rango de bit-canal (1 = más confiable, N = más ruidoso)")
axes[1].set_ylabel("BER empírico [escala log]")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Módulo 6: Persistencia y Reutilización de Perfiles de Confiabilidad (`BER_bits`)

Separar la **construcción del código** (estimación de `BER_bits`) de su **evaluación BER/BLER** permite ahorrar tiempo computacional y habilitar experimentos de **desajuste de construcción** (e.g., cargar `BER_bits` diseñado para AWGN y evaluar su pérdida sobre un canal Suzuki).

Convención de nombres (compatible con `gui.py`):
```text
./results_{N}/YYMMDD_{canal}_{ecualizador}_{snr}SNR_BER_bits_{N}_{K}_{num_iter}iter.csv
```

In [ ]:
# Guardado del vector BER_bits recién generado
assert len(BER_bits) == N and np.all(np.isfinite(BER_bits)), (
    f"⚠️ BER_bits está incompleto ({np.sum(np.isfinite(BER_bits))}/{N} valores finitos)."
)

output_dir = f"results_{N}"
os.makedirs(output_dir, exist_ok=True)
fecha = datetime.now().strftime("%y%m%d")
filename_ber = os.path.join(
    output_dir,
    f"{fecha}_{canal}_{ecualizador}_{snr_simulacion}SNR_BER_bits_{N}_{K}_{num_iter}iter.csv",
)
pd.DataFrame({"BER bits": BER_bits}).to_csv(filename_ber, index=True, encoding="utf-8-sig")
print(f"💾 Perfil BER_bits guardado en: {filename_ber}")

In [ ]:
# Carga opcional de un archivo CSV de BER_bits previamente calculado
# Cambie CARGAR_DESDE_CSV = True e indique la ruta deseada para reutilizar un diseño previo.
CARGAR_DESDE_CSV = False
CSV_BER_BITS_PATH = os.path.join(f"results_{N}", f"YYMMDD_canal_eq_SNR_BER_bits_{N}_{K}_1000iter.csv")  # ejemplo: reemplazar por archivo real

if CARGAR_DESDE_CSV:
    df_loaded = pd.read_csv(CSV_BER_BITS_PATH, encoding="utf-8-sig")
    if "BER bits" not in df_loaded.columns:
        raise ValueError("El archivo CSV no contiene la columna 'BER bits'.")

    col_ber = pd.to_numeric(df_loaded["BER bits"], errors="coerce")
    if col_ber.isna().any():
        raise ValueError("La columna 'BER bits' contiene valores vacíos o no numéricos.")

    BER_bits = col_ber.to_numpy(dtype=np.float64)
    if not np.all(np.isfinite(BER_bits)) or np.any((BER_bits < 0.0) | (BER_bits > 1.0)):
        raise ValueError("Los valores de 'BER bits' deben ser finitos y pertenecer a [0.0, 1.0].")
    if len(BER_bits) != N:
        raise ValueError(
            f"Desajuste de longitud: el CSV tiene {len(BER_bits)} bit-canales pero N={N}. "
            "Actualice N en el Módulo 1 o seleccione un CSV compatible."
        )

    zero_ber_count = int(np.count_nonzero(BER_bits == 0.0))
    if zero_ber_count > K:
        print(
            f"⚠️ Advertencia: {zero_ber_count}/{N} bit-canales tienen BER = 0.0 (> K={K}). "
            "Los empates se resuelven con Polarization Weight (beta = 2^(1/4))."
        )
    indices_ordenados = rank_polar_bit_channels(BER_bits)
    bits_info = indices_ordenados[:K]
    bits_congelados = indices_ordenados[-(N - K):]
    print(f"📂 Cargados {len(BER_bits)} bit-canales desde: {CSV_BER_BITS_PATH}")
else:
    print("ℹ️ CARGAR_DESDE_CSV = False (se conserva el vector BER_bits actual en memoria).")

---
## Módulo 7: Evaluación Comparativa de Esquemas de Codificación (BER vs. $E_b/N_0$)

En este módulo contrastamos cuatro arquitecturas sobre exactamente el mismo canal, bloque de coherencia, interleaver, error CSI y ecualizador:

1. **Sin Código de Canal (`SinCodigo`, $R = 1.0$):** Referencia sin redundancia; evalúa decisiones duras a partir de LLRs blandos (`soft_estimates=True`).
2. **Código Polar + SCL (`CodigoPolar`, $R = K/N$):** Decodificador *Successive Cancellation List* de tamaño $L$ y complejidad $\mathcal{O}(LN\log N)$. Mejora drásticamente a SC ($L=1$), aunque su espectro de distancias mínimas puede limitar la pendiente en régimen de alta SNR.
3. **Código Polar + CRC / CA-SCL (`CodigoPolarCRC`, 3GPP TS 38.212, $R_{\text{efec}} = (K - L_{\text{CRC}})/N$):** Concatena un código externo CRC para descartar trayectorias inválidas dentro de la lista $L$, elevando efectivamente la distancia mínima del código al costo de una penalización de tasa $L_{\text{CRC}}/N$ (crítica cuando $N$ es pequeño).
4. **Código LDPC 5G NR (`CodigoLDPC`, 3GPP TS 38.212, $R = K/N$):** Códigos cuasi-cíclicos de matriz de paridad dispersa ($\mathbf{H}\mathbf{c}^T = \mathbf{0}$) decodificados iterativamente mediante *Belief Propagation* (BP). Históricamente, CA-SCL domina en bloques cortos ($N \le 512$), mientras LDPC compite y supera en eficiencia de decodificación paralela a bloques largos.

Todos los sistemas heredan de una clase base común (`BaseSystemBlock`) que encapsula el pipeline de canal y recepción para evitar discrepancias entre curvas.

In [ ]:
# =====================================================================
# BLOQUES DEL SISTEMA SIONNA (ARQUITECTURA MODULAR COMPARTIDA)
# =====================================================================

class BaseSystemBlock(sionna.phy.Block):
    """Clase base común que encapsula canal, error CSI, ecualización, demapeo y desintercalado."""

    def __init__(self, params):
        super().__init__()
        self.p = dict(params)

    def apply_channel_and_eq(self, x, ebno_db, coderate):
        return apply_channel_and_equalizer(x, ebno_db, coderate=coderate, params=self.p)


class SinCodigo(BaseSystemBlock):
    """Línea base sin codificación de canal (R = 1.0)."""

    def call(self, batch_size, ebno_db):
        b = datos([batch_size, self.p["K"]])
        x = mapeador(interleaver(b))
        llr = self.apply_channel_and_eq(x, ebno_db, coderate=1.0)
        return b, llr


class CodigoPolar(BaseSystemBlock):
    """Código Polar con decodificación Successive Cancellation List (SCL)."""

    def __init__(self, params, frozen_pos):
        super().__init__(params)
        self.polar_codif = sionna.phy.fec.polar.encoding.PolarEncoder(frozen_pos, self.p["N"])
        self.polar_decodif = sionna.phy.fec.polar.decoding.PolarSCLDecoder(
            frozen_pos, self.p["N"], list_size=self.p["list_scl"]
        )

    def call(self, batch_size, ebno_db):
        b = datos([batch_size, self.p["K"]])
        x = mapeador(interleaver(self.polar_codif(b)))
        llr = self.apply_channel_and_eq(x, ebno_db, coderate=self.p["K"] / self.p["N"])
        return b, self.polar_decodif(llr)


class CodigoPolarCRC(BaseSystemBlock):
    """Código Polar concatenado con CRC 3GPP 5G NR (CA-SCL)."""

    def __init__(self, params, frozen_pos):
        super().__init__(params)
        self.crc_enc = sionna.phy.fec.crc.CRCEncoder(crc_degree=self.p["crc_degree"])
        self.crc_len = self.crc_enc.crc_length
        if self.p["K"] <= self.crc_len:
            raise ValueError(f"K ({self.p['K']}) debe ser mayor que la longitud del CRC ({self.crc_len}).")
        self.polar_codif = sionna.phy.fec.polar.encoding.PolarEncoder(frozen_pos, self.p["N"])
        self.polar_decodif = sionna.phy.fec.polar.decoding.PolarSCLDecoder(
            frozen_pos, self.p["N"], list_size=self.p["list_scl"], crc_degree=self.p["crc_degree"]
        )

    def call(self, batch_size, ebno_db):
        k_efec = self.p["K"] - self.crc_len
        b = datos([batch_size, k_efec])
        bits_codif = self.polar_codif(self.crc_enc(b))
        x = mapeador(interleaver(bits_codif))
        llr = self.apply_channel_and_eq(x, ebno_db, coderate=k_efec / self.p["N"])
        b_hat = self.polar_decodif(llr)[..., :k_efec]
        return b, b_hat


class CodigoLDPC(BaseSystemBlock):
    """Código LDPC 5G NR (3GPP TS 38.212) con decodificación Belief Propagation."""

    def __init__(self, params):
        super().__init__(params)
        self.ldpc_codif = sionna.phy.fec.ldpc.encoding.LDPC5GEncoder(
            self.p["K"], self.p["N"], self.p["bitsxsimb"]
        )
        self.ldpc_decodif = sionna.phy.fec.ldpc.decoding.LDPC5GDecoder(self.ldpc_codif)

    def call(self, batch_size, ebno_db):
        b = datos([batch_size, self.p["K"]])
        x = mapeador(interleaver(self.ldpc_codif(b)))
        llr = self.apply_channel_and_eq(x, ebno_db, coderate=self.p["K"] / self.p["N"])
        return b, self.ldpc_decodif(llr)


# Inicializa contenedor de gráficas PlotBER y registro numérico para análisis posterior
sim_params = get_sim_params()
ebno_grid = np.linspace(EBNO_DB_MIN, EBNO_DB_MAX, EBNO_RANGO)
csi_eff_title = sim_params["csi_err_var"] if sim_params["canal"] not in CANALES_SIN_FADING else 0.0
csi_suffix = f", σe²={csi_eff_title}" if csi_eff_title > 0.0 else ""
ber_plots = sionna.phy.utils.PlotBER(
    f"Comparación BER — Canal {sim_params['canal'].capitalize()} "
    f"(N={sim_params['N']}, K={sim_params['K']}, Eq={sim_params['ecualizador'].upper()}{csi_suffix})"
)
resultados_curvas = {}


def registrar_resultado(nombre, sim_out):
    """Almacena BER y BLER retornados por PlotBER.simulate para métricas cuantitativas."""
    if isinstance(sim_out, tuple) and len(sim_out) >= 2:
        resultados_curvas[nombre] = {
            "ebno_db": ebno_grid.copy(),
            "ber": np.asarray(sim_out[0], dtype=np.float64),
            "bler": np.asarray(sim_out[1], dtype=np.float64),
        }

In [ ]:
# 7.1 Simulación: Línea base Sin Código de Canal
sin_codigo = SinCodigo(sim_params)
res_uncoded = ber_plots.simulate(
    sin_codigo,
    ebno_dbs=ebno_grid,
    batch_size=lote_berplot,
    num_target_block_errors=TARGET_BLOCK_ERRORS,
    legend="Sin código",
    soft_estimates=True,
    max_mc_iter=MAX_MC_ITER_BER,
    show_fig=True,
    forward_keyboard_interrupt=False,
)
registrar_resultado("Sin código", res_uncoded)

In [ ]:
# 7.2 Simulación: Código Polar (SCL)
assert "bits_congelados" in globals() and len(bits_congelados) == sim_params["N"] - sim_params["K"], (
    "Ejecute el Módulo 5 (Monte Carlo) o cargue un CSV compatible en el Módulo 6 antes de simular Polar."
)
codigo_polar = CodigoPolar(sim_params, bits_congelados)
res_polar = ber_plots.simulate(
    codigo_polar,
    ebno_dbs=ebno_grid,
    batch_size=lote_berplot,
    num_target_block_errors=TARGET_BLOCK_ERRORS,
    legend=f"Polar (SCL L={sim_params['list_scl']})",
    soft_estimates=False,
    max_mc_iter=MAX_MC_ITER_BER,
    show_fig=True,
    forward_keyboard_interrupt=False,
)
registrar_resultado(f"Polar (SCL L={sim_params['list_scl']})", res_polar)

In [ ]:
# 7.3 Simulación: Código Polar + CRC (CA-SCL 3GPP 5G NR)
assert "bits_congelados" in globals() and len(bits_congelados) == sim_params["N"] - sim_params["K"], (
    "Ejecute el Módulo 5 (Monte Carlo) o cargue un CSV compatible en el Módulo 6 antes de simular Polar + CRC."
)
if sim_params["K"] > crc_long:
    codigo_polar_crc = CodigoPolarCRC(sim_params, bits_congelados)
    label_crc = f"Polar + {sim_params['crc_degree']} (CA-SCL L={sim_params['list_scl']})"
    res_polar_crc = ber_plots.simulate(
        codigo_polar_crc,
        ebno_dbs=ebno_grid,
        batch_size=lote_berplot,
        num_target_block_errors=TARGET_BLOCK_ERRORS,
        legend=label_crc,
        soft_estimates=False,
        max_mc_iter=MAX_MC_ITER_BER,
        show_fig=True,
        forward_keyboard_interrupt=False,
    )
    registrar_resultado(label_crc, res_polar_crc)
else:
    print(f"⚠️ Omitiendo Polar + {sim_params['crc_degree']}: K ({sim_params['K']}) <= longitud CRC ({crc_long}).")

In [ ]:
# 7.4 Simulación: Código LDPC 5G NR (Belief Propagation)
try:
    codigo_ldpc = CodigoLDPC(sim_params)
except (ValueError, AssertionError) as ldpc_err:
    print(f"⚠️ Omitiendo LDPC 5G NR para (K={sim_params['K']}, N={sim_params['N']}): {ldpc_err}")
else:
    res_ldpc = ber_plots.simulate(
        codigo_ldpc,
        ebno_dbs=ebno_grid,
        batch_size=lote_berplot,
        num_target_block_errors=TARGET_BLOCK_ERRORS,
        legend="LDPC 5G NR (BP)",
        soft_estimates=False,
        max_mc_iter=MAX_MC_ITER_BER,
        show_fig=True,
        forward_keyboard_interrupt=False,
    )
    registrar_resultado("LDPC 5G NR (BP)", res_ldpc)

In [ ]:
# =====================================================================
# RESUMEN CUANTITATIVO: GANANCIA DE CODIFICACIÓN, BLER Y EXPORTACIÓN CSV
# =====================================================================

def _interp_ebno_at_target(ebno_arr, err_arr, target):
    """Interpola linealmente en log10(BER) el Eb/N0 [dB] donde se cruza el umbral target."""
    valid = np.isfinite(err_arr) & (err_arr > 0.0)
    x = ebno_arr[valid]
    y = np.log10(err_arr[valid])
    log_t = np.log10(target)
    for i in range(len(x) - 1):
        if (y[i] >= log_t >= y[i + 1]) or (y[i] <= log_t <= y[i + 1]):
            if y[i + 1] == y[i]:
                return float(x[i])
            frac = (log_t - y[i]) / (y[i + 1] - y[i])
            return float(x[i] + frac * (x[i + 1] - x[i]))
    return np.nan


if resultados_curvas:
    ref_ebno_1e3 = (
        _interp_ebno_at_target(resultados_curvas["Sin código"]["ebno_db"], resultados_curvas["Sin código"]["ber"], 1e-3)
        if "Sin código" in resultados_curvas else np.nan
    )
    ref_ebno_1e4 = (
        _interp_ebno_at_target(resultados_curvas["Sin código"]["ebno_db"], resultados_curvas["Sin código"]["ber"], 1e-4)
        if "Sin código" in resultados_curvas else np.nan
    )

    filas = []
    for nombre, d in resultados_curvas.items():
        eb_1e3 = _interp_ebno_at_target(d["ebno_db"], d["ber"], 1e-3)
        eb_1e4 = _interp_ebno_at_target(d["ebno_db"], d["ber"], 1e-4)
        bler_1e2 = _interp_ebno_at_target(d["ebno_db"], d["bler"], 1e-2)
        gain_1e3 = (ref_ebno_1e3 - eb_1e3) if np.isfinite(ref_ebno_1e3) and np.isfinite(eb_1e3) else np.nan
        gain_1e4 = (ref_ebno_1e4 - eb_1e4) if np.isfinite(ref_ebno_1e4) and np.isfinite(eb_1e4) else np.nan
        filas.append({
            "Esquema": nombre,
            "Eb/N0 @ BER=1e-3 [dB]": round(eb_1e3, 2) if np.isfinite(eb_1e3) else np.nan,
            "Ganancia @ 1e-3 [dB]": round(gain_1e3, 2) if np.isfinite(gain_1e3) else np.nan,
            "Eb/N0 @ BER=1e-4 [dB]": round(eb_1e4, 2) if np.isfinite(eb_1e4) else np.nan,
            "Ganancia @ 1e-4 [dB]": round(gain_1e4, 2) if np.isfinite(gain_1e4) else np.nan,
            "Eb/N0 @ BLER=1e-2 [dB]": round(bler_1e2, 2) if np.isfinite(bler_1e2) else np.nan,
        })

    df_resumen = pd.DataFrame(filas)
    try:
        display(df_resumen)
    except NameError:
        print(df_resumen.to_string(index=False))

    # Gráfica complementaria de BLER vs Eb/N0
    fig, ax = plt.subplots(figsize=(9, 5))
    marcadores = ["o-", "s-", "^-", "D-"]
    min_bler_pos = 1e-5
    for idx_c, (nombre, d) in enumerate(resultados_curvas.items()):
        mask = np.isfinite(d["bler"]) & (d["bler"] > 0.0)
        if np.any(mask):
            min_bler_pos = min(min_bler_pos, float(np.min(d["bler"][mask])))
            ax.semilogy(d["ebno_db"][mask], d["bler"][mask], marcadores[idx_c % len(marcadores)], lw=2, ms=6, label=nombre)
    ylim_low = min(1e-5, max(1e-9, 10.0 ** np.floor(np.log10(min_bler_pos))))
    ax.set_title(
        f"Comparación BLER vs Eb/N0 — Canal {sim_params['canal'].capitalize()} "
        f"(N={sim_params['N']}, K={sim_params['K']}, Eq={sim_params['ecualizador'].upper()}{csi_suffix})"
    )
    ax.set_xlabel(r"$E_b/N_0$ [dB]")
    ax.set_ylabel("BLER")
    ax.set_ylim(ylim_low, 1.0)
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Exportación opcional de las curvas BER y BLER a CSV
    EXPORTAR_CURVAS_CSV = False
    if EXPORTAR_CURVAS_CSV:
        out_dir = f"results_{sim_params['N']}"
        os.makedirs(out_dir, exist_ok=True)
        fecha = datetime.now().strftime("%y%m%d")
        csv_curves = os.path.join(
            out_dir,
            f"{fecha}_{sim_params['canal']}_{sim_params['ecualizador']}_BER_curves_{sim_params['N']}_{sim_params['K']}.csv",
        )
        export_dict = {"Eb/N0 [dB]": ebno_grid}
        for nombre, d in resultados_curvas.items():
            export_dict[f"{nombre} (BER)"] = d["ber"]
            export_dict[f"{nombre} (BLER)"] = d["bler"]
        pd.DataFrame(export_dict).to_csv(csv_curves, index=False, encoding="utf-8-sig")
        print(f"💾 Curvas BER/BLER exportadas en: {csv_curves}")
else:
    print("ℹ️ No hay curvas registradas todavía.")